# 03 | Sales-Tier Modeling

Create console-generation and sales-tier fields, then compare Logistic Regression and Random Forest using only information available before release.

`Sales_Tier` is derived from global-sales tertile cutoffs. It is the outcome, not a predictor. The model uses only `Platform`, `Genre`, `Publisher`, and `Year`; encoders and scaling are fit inside each training fold to avoid leakage. Stratified splitting and class-weighted models help with class imbalance, while macro metrics give each tier equal weight.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.features import add_console_generation, add_sales_tier
from src.model import evaluate_models

CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'vgsales_clean.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not CLEAN_PATH.exists():
    raise FileNotFoundError('Run 01_data_cleaning.ipynb first')

data = pd.read_csv(CLEAN_PATH)
data = add_console_generation(data)
data = add_sales_tier(data)
display(data[['Year', 'Console_Generation', 'Global_Sales', 'Sales_Tier']].head())
display(data['Sales_Tier'].value_counts().rename('titles'))

## Holdout and cross-validation

An 80/20 stratified split reserves unseen titles for final evaluation. Five-fold stratified cross-validation is run on the training partition. Reported precision, recall, and F1 are macro-averaged.

In [ ]:
results = evaluate_models(data, test_size=0.2, cv_splits=5, random_state=42)
print('Held-out test metrics')
display(results['metrics'].round(3))
print('Mean training-fold cross-validation metrics')
display(results['cv_scores'].round(3))
print('Predictors:', ', '.join(results['features']))
print('Tier order:', results['classes'])

## Confusion matrices

Rows are actual sales tiers and columns are predicted tiers. Use these alongside macro scores to see which tiers are most often confused.

In [ ]:
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, len(results['confusion_matrices']), figsize=(12, 5))
for ax, (model_name, matrix) in zip(axes, results['confusion_matrices'].items()):
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=results['classes'], yticklabels=results['classes'], ax=ax)
    ax.set(title=model_name, xlabel='Predicted tier', ylabel='Actual tier')
fig.suptitle('Held-out confusion matrices', y=1.03)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'model_confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved confusion matrix figure to {FIGURE_DIR.relative_to(PROJECT_ROOT)}')